In [22]:
# === SETUP ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

os.makedirs("../plots", exist_ok=True)

# load analysis dataset ---
df = pd.read_csv("C:/Users/lalib/AppData/Local/Programs/data/analysis_dataset.csv")
df["datetime"] = pd.to_datetime(df["datetime"], utc=True)

# merge TTF gas price (daily → hourly, forward-filled)
ttf = pd.read_csv("C:/Users/lalib/AppData/Local/Programs/data/ttf.csv")
ttf.columns = ttf.columns.str.strip().str.lower()
ttf["date"] = pd.to_datetime(ttf["date"]).dt.date
df["date"] = df["datetime"].dt.date
df = df.merge(ttf, on="date", how="left")
df["ttf_eur_mwh"] = df["ttf_eur_mwh"].ffill()

# time features in LOCAL time (SQL strftime returns UTC)
df["local"] = df["datetime"].dt.tz_convert("Europe/Athens")
df["hour"] = df["local"].dt.hour
df["month"] = df["local"].dt.month
df["day_of_week"] = df["local"].dt.dayofweek
df["year"] = df["local"].dt.year
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

def get_season(m):
    if m in [12, 1, 2]: return "Winter"
    elif m in [3, 4, 5]: return "Spring"
    elif m in [6, 7, 8]: return "Summer"
    else: return "Fall"

df["season"] = df["month"].apply(get_season)

# --- sort by time (essential for lag features later) ---
df = df.sort_values("datetime").reset_index(drop=True)

print("Shape:", df.shape)
print("Range:", df["local"].min(), "→", df["local"].max())

Shape: (35065, 20)
Range: 2022-08-01 00:00:00+03:00 → 2026-08-01 00:00:00+03:00


In [23]:
df=df[df["local"]<"2025-08-01"].copy()
print("Modelling window:",df["local"].min(), "→" ,df ["local"].max(), "|", len(df), "rows" )
df["residual_load"]=df["load"]-df["solar"]-df["wind"]

df["price_lag_24"]=df["price"].shift(24)
df["price_lag_168"]=df["price"].shift(168)
df["load_lag_24"]=df["load"].shift(24)


df["price_roll_24"]=df["price"].shift(1).rolling(24).mean()
df["ttf_roll_7d"]=df["ttf_eur_mwh"].shift(1).rolling(168).mean()

df["hour_sin"]=np.sin(2*np.pi*df["hour"]/24)
df["hour_cos"]=np.cos(2*np.pi*df["hour"]/24)

df["crisis"]=(df["local"]< "2023-07-01").astype(int)

print("\nMissing per feature:")
feats = ["price","load","solar","wind","gas","ttf_eur_mwh","residual_load",
         "price_lag_24","price_lag_168","price_roll_24","hour_sin","hour_cos","crisis"]

print(df[feats].isna().sum())


Modelling window: 2022-08-01 00:00:00+03:00 → 2025-07-31 23:00:00+03:00 | 26304 rows

Missing per feature:
price              0
load               3
solar              8
wind               8
gas               10
ttf_eur_mwh        3
residual_load      9
price_lag_24      24
price_lag_168    168
price_roll_24     24
hour_sin           0
hour_cos           0
crisis             0
dtype: int64


In [24]:
model_df=pd.read_csv("C:/Users/lalib/AppData/Local/Programs/data/model_dataset.csv")
model_df["local"] = pd.to_datetime(model_df["local"], utc=True).dt.tz_convert("Europe/Athens")
model_df["datetime"] = pd.to_datetime(model_df["datetime"], utc=True)

In [25]:
price_by_year=df.groupby("year")["price"].mean().round(2)
print(price_by_year)
print("\nFormatter for FINDINGS:")
for year, price in price_by_year.items():\
    print(f"{int(year)}: €{price:.2f}")


year
2022    318.11
2023    119.12
2024    100.88
2025    106.98
Name: price, dtype: float64

Formatter for FINDINGS:
2022: €318.11
2023: €119.12
2024: €100.88
2025: €106.98


In [26]:
price_by_hour=df[df["year"]>= 2024].groupby("hour")["price"].mean().round(2)

trough_hour=price_by_hour.idxmin()
trough_price=price_by_hour.min()
peak_hour=price_by_hour.idxmax()
peak_price=price_by_hour.max()
ratio=(peak_price/trough_price).round(2)


print(f"\nTrough: {trough_hour}:00 = €{trough_price:.2f}")
print(f"Peak: {peak_hour}:00 = €{peak_price:.2f}")
print(f"Ratio (peak/trough): {ratio}x")


Trough: 12:00 = €55.66
Peak: 20:00 = €182.22
Ratio (peak/trough): 3.27x


In [27]:
d=model_df.dropna(subset=["residual_load","price"]).copy()
d["rl_bin"]=pd.cut(d["residual_load"],bins=20)
binned=d.groupby("rl_bin",observed=True).agg(
    rl=("residual_load","mean"),
    price_med=("price","median"),
    price_p90=("price",lambda x: x.quantile(0.90))

).reset_index(drop=True)

print(binned.round(1))


binned["spread"]=binned["price_p90"]-binned["price_med"]
print("\nSpread (p90 - median):")
print(binned[["rl","spread"]].round(1))

        rl  price_med  price_p90
0  -1876.0        2.6       27.0
1  -1193.5       30.5       66.7
2   -744.3       40.0       88.9
3   -179.9       44.5      103.9
4    381.1       50.5      102.8
5    932.3       67.2      115.3
6   1449.1       79.6      134.8
7   2000.4       89.0      163.0
8   2541.9       99.6      190.0
9   3092.3      107.1      220.8
10  3632.0      113.8      256.4
11  4168.9      124.8      329.9
12  4717.4      134.7      340.2
13  5259.3      145.1      364.0
14  5800.3      163.2      424.5
15  6343.9      177.5      372.7
16  6889.8      190.6      426.9
17  7434.9      178.9      387.0
18  7997.7      189.9      498.1
19  8506.0      227.3      675.0

Spread (p90 - median):
        rl  spread
0  -1876.0    24.4
1  -1193.5    36.2
2   -744.3    48.9
3   -179.9    59.4
4    381.1    52.3
5    932.3    48.0
6   1449.1    55.2
7   2000.4    74.0
8   2541.9    90.4
9   3092.3   113.7
10  3632.0   142.7
11  4168.9   205.1
12  4717.4   205.5
13  5259.3   219.

In [28]:
d=df[df["year"]>=2024].copy()

weekday=d[d["is_weekend"]==0].groupby("hour")["price"].mean().round(2)
weekend=d[d["is_weekend"]==1].groupby("hour")["price"].mean().round(2)

comparison=pd.DataFrame({
    "hour":range(24),
    "weekday":weekday.values,
    "weekend":weekend.values
    })

comparison["difference"]=(comparison["weekend"]-comparison["weekday"]).round(2) 
print(comparison.to_string(index=False)) 

print(f"\nAverage weekday price: €{weekday.mean():.2f}")
print(f"\Average weekend price: €{weekend.mean():.2f}")
print(f"Weekend discount: {((1 - weekend.mean()/weekday.mean())*100):.1f}%")

 hour  weekday  weekend  difference
    0   103.79   102.84       -0.95
    1    97.78   100.26        2.48
    2    93.40    95.50        2.10
    3    89.05    91.02        1.97
    4    87.33    88.09        0.76
    5    89.77    87.14       -2.63
    6   100.67    87.79      -12.88
    7   122.15    88.38      -33.77
    8   133.22    84.07      -49.15
    9   113.84    70.26      -43.58
   10    88.30    52.68      -35.62
   11    71.38    41.00      -30.38
   12    63.72    35.34      -28.38
   13    62.99    37.46      -25.53
   14    67.05    40.16      -26.89
   15    78.00    48.27      -29.73
   16    97.64    61.47      -36.17
   17   120.84    83.35      -37.49
   18   141.09   109.52      -31.57
   19   168.50   127.94      -40.56
   20   197.23   144.31      -52.92
   21   185.04   147.31      -37.73
   22   143.70   130.06      -13.64
   23   119.65   113.04       -6.61

Average weekday price: €109.84
\Average weekend price: €86.14
Weekend discount: 21.6%


<>:16: SyntaxWarning: invalid escape sequence '\A'
<>:16: SyntaxWarning: invalid escape sequence '\A'
C:\Users\lalib\AppData\Local\Temp\ipykernel_22872\4037190973.py:16: SyntaxWarning: invalid escape sequence '\A'
  print(f"\Average weekend price: €{weekend.mean():.2f}")


In [29]:
d = df.dropna(subset=["ttf_eur_mwh", "price"]).copy()

# Pooled hourly
r_hourly = d["ttf_eur_mwh"].corr(d["price"], method="pearson")
print(f"Hourly TTF–price correlation (all years): {r_hourly:.3f}")

# 2024+ only
d24 = d[d["year"] >= 2024]
r_hourly_24 = d24["ttf_eur_mwh"].corr(d24["price"], method="pearson")
print(f"Hourly TTF–price correlation (2024+):    {r_hourly_24:.3f}")

# Monthly for comparison
monthly = df.groupby(["year", "month"])[["ttf_eur_mwh", "price"]].mean().dropna()
r_monthly = monthly["ttf_eur_mwh"].corr(monthly["price"], method="pearson")
print(f"Monthly TTF–price correlation:           {r_monthly:.3f}")

Hourly TTF–price correlation (all years): 0.779
Hourly TTF–price correlation (2024+):    0.321
Monthly TTF–price correlation:           0.971


In [30]:
neg_by_year = df[df["price"] < 0].groupby("year").size()
print("Negative price hours per year:")
print(neg_by_year)
print(f"\nTotal: {neg_by_year.sum()} hours")

Negative price hours per year:
year
2022     1
2024    11
2025    12
dtype: int64

Total: 24 hours


In [31]:
spread_by_year = df.groupby("year").apply(
    lambda d: d[d["hour"].between(19, 22)]["price"].mean() - d[d["hour"].between(11, 15)]["price"].mean()
).round(2)

print("Evening (19–22h) minus midday (11–15h) price spread:")
print(spread_by_year)

Evening (19–22h) minus midday (11–15h) price spread:
year
2022    127.29
2023     65.14
2024     98.04
2025    110.77
dtype: float64


In [32]:
solar_by_year = df[(df["hour"].between(11, 15)) & (df["year"] >= 2023)].groupby("year")["solar"].mean().round(0)
print("Average midday (11–15h) solar generation (2023–2025):")
print(solar_by_year)

Average midday (11–15h) solar generation (2023–2025):
year
2023    2964.0
2024    3833.0
2025    3838.0
Name: solar, dtype: float64
